# 📊 Análisis de Correspondencia Múltiple (ACM / MCA)
## *Hábitos de sueño y rendimiento: perfiles estudiantiles — Exposición Completa*

---

| Sección | Tema | Tiempo estimado |
|---------|------|-----------------|
| 0 | Motivación: ¿por qué MCA? | ~5 min |
| 1 | Formalismo matemático | ~8 min |
| 2 | Contexto y datos del estudio | ~5 min |
| 3 | Preprocesamiento y limpieza | ~7 min |
| 4 | Implementación del MCA | ~10 min |
| 5 | Visualizaciones e interpretación | ~10 min |
| 6 | Perfiles estudiantiles y conclusiones | ~5 min |

---

# 🔍 Sección 0 — ¿Por qué Análisis de Correspondencia Múltiple?

## El problema de las variables categóricas

Imagina que encuestas a 64 estudiantes sobre sus hábitos de sueño y rendimiento académico. Las respuestas no son números: son categorías como *"a veces"*, *"frecuentemente"*, *"mucho"*.

**¿Cómo analizar estas relaciones?**

| Técnica | ¿Cuándo usarla? | ¿Por qué no aquí? |
|---------|-----------------|-------------------|
| Correlación de Pearson | Variables numéricas continuas | Nuestras variables son categóricas |
| Regresión Lineal | Variable respuesta numérica | No hay números |
| PCA | Variables numéricas | No funciona con categorías directamente |
| **MCA** ✅ | **Múltiples variables categóricas** | **¡Exactamente nuestro caso!** |

## ¿Qué hace MCA?

> MCA es la extensión de PCA para **variables categóricas**. Convierte tablas de encuestas en un **mapa visual** donde:
> - Categorías **cercanas** tienden a aparecer juntas en los mismos individuos
> - Las **dimensiones** resumen los principales patrones de variación
> - Se pueden identificar **perfiles** de individuos similares

```
Encuesta (64 estudiantes × 6 variables categóricas)  →  MCA  →  Mapa 2D de perfiles
       [difícil de interpretar directamente]                   [fácil de visualizar]
```

## Analogía intuitiva

Piensa en un **mapa geográfico**: las ciudades cercanas comparten clima, cultura, etc. De forma similar, el MCA crea un **mapa de categorías** donde las que aparecen juntas en los mismos individuos quedan cerca en el gráfico.

# 📐 Sección 1 — Formalismo Matemático del MCA

---

## 1.1 La Matriz Indicadora (Burt)

Dado un dataset con $N$ individuos y $Q$ variables categóricas, construimos la **matriz indicadora** $Z$:

$$Z \in \{0, 1\}^{N \times K}$$

donde $K = \sum_{q=1}^Q K_q$ es el total de categorías en todas las variables. Cada fila tiene exactamente **$Q$ unos** (uno por variable).

**Ejemplo:** Si SLEEP tiene 4 categorías y FATIGA tiene 3, $K_1 = 4$, $K_2 = 3$, ...

---

## 1.2 El Análisis de Correspondencias

MCA aplica **Análisis de Correspondencias Simple (CA)** sobre la matriz $Z$.

**Paso 1:** Calcular la matriz de proporciones:
$$P = \frac{Z}{N_{total}}, \quad N_{total} = \sum_{i,j} z_{ij}$$

**Paso 2:** Calcular las masas de filas y columnas:
$$r_i = \frac{\sum_j p_{ij}}{1}, \quad c_j = \frac{\sum_i p_{ij}}{1}$$

**Paso 3:** Construir la matriz estandarizada para SVD:
$$S = D_r^{-1/2} \left(P - r c^T\right) D_c^{-1/2}$$

donde $D_r = \text{diag}(r)$ y $D_c = \text{diag}(c)$.

---

## 1.3 Descomposición SVD

Aplicamos SVD a $S$:
$$S = U \Sigma V^T$$

| Elemento | Dimensión | Significado |
|----------|-----------|-------------|
| $\sigma_k^2$ | escalar | **Eigenvalor** = varianza (inercia) explicada por la dimensión $k$ |
| Columnas de $U$ | $N \times K$ | Direcciones en el espacio de individuos |
| Filas de $V^T$ | $K \times K$ | Direcciones en el espacio de categorías |

---

## 1.4 Coordenadas Factoriales

**Coordenadas de los individuos (filas):**
$$F = D_r^{-1/2} U \Sigma$$

**Coordenadas de las categorías (columnas):**
$$G = D_c^{-1/2} V \Sigma$$

---

## 1.5 Inercia Explicada

La **inercia total** de MCA es:
$$I_{total} = \frac{K}{Q} - 1$$

La inercia explicada por la dimensión $k$:
$$\% \text{inercia}_k = \frac{\sigma_k^2}{I_{total}} \times 100$$

> **Nota importante:** En MCA, los porcentajes de varianza explicada suelen ser menores que en PCA porque la inercia total es más grande al trabajar con variables dummy. Valores del 50-70% con 2-4 dimensiones ya son considerados buenos.

# 🎓 Sección 2 — Contexto y Datos del Estudio

## El dataset

Datos recolectados por estudiantes de **Probabilidad y Estadística I** sobre hábitos de sueño y rendimiento académico.

- **64 estudiantes** de múltiples carreras de la universidad
- **16 variables** (categóricas y numéricas)
- Variables organizadas en 3 dimensiones:

| Dimensión | Variables |
|-----------|----------|
| 🎓 **Académica** | SEMESTRE, CARRERA, CRÉDITOS, RETIRO |
| 😴 **Personal/Bienestar** | HORAS, SLEEP, CONCENTRACIÓN, PUNTUALIDAD, FATIGA, CALIDAD |
| 💰 **Socioeconómica** | BECA, DOBLECARRERA, ACTIVIDADES, GÉNERO |

In [ ]:
# ============================================================
# LIBRERÍAS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.35,
    'font.size': 12
})

# ============================================================
# IMPLEMENTACIÓN MCA DESDE CERO (SVD)
# ============================================================
class MCA_SVD:
    """
    Análisis de Correspondencia Múltiple implementado
    mediante Descomposición en Valores Singulares (SVD).
    No requiere librerías externas más allá de numpy.
    """
    def __init__(self, n_components=5):
        self.n_components = n_components
        
    def fit_transform(self, data_categorical):
        """Ajusta el MCA y retorna coordenadas de filas."""
        # 1. Construir matriz indicadora Z (one-hot)
        self.Z_ = pd.get_dummies(data_categorical)
        self.col_names_ = list(self.Z_.columns)
        Z = self.Z_.values.astype(float)
        self.n_, self.K_ = Z.shape
        self.Q_ = data_categorical.shape[1]  # número de variables
        
        # 2. Normalizar → tabla de proporciones P
        N_total = Z.sum()
        P = Z / N_total
        
        # 3. Masas de filas y columnas
        self.r_ = P.sum(axis=1)          # vector de masas de filas
        self.c_ = P.sum(axis=0)          # vector de masas de columnas
        
        # 4. Matriz estandarizada para SVD
        Dr_inv_sqrt = np.diag(1.0 / np.sqrt(self.r_))
        Dc_inv_sqrt = np.diag(1.0 / np.sqrt(self.c_))
        S = Dr_inv_sqrt @ (P - np.outer(self.r_, self.c_)) @ Dc_inv_sqrt
        
        # 5. SVD
        U, sv, Vt = np.linalg.svd(S, full_matrices=False)
        # Eliminar la primera componente trivial
        U, sv, Vt = U[:, 1:], sv[1:], Vt[1:, :]
        
        # 6. Eigenvalores e inercia
        self.eigenvalues_ = sv**2
        I_total = (self.K_ / self.Q_) - 1   # inercia total teórica
        self.explained_inertia_ = self.eigenvalues_ / I_total
        self.cumulative_inertia_ = np.cumsum(self.explained_inertia_)
        
        # 7. Coordenadas factoriales
        k = self.n_components
        # Individuos (filas)
        self.row_coords_ = Dr_inv_sqrt @ U[:, :k] * sv[:k]
        # Categorías (columnas)
        self.col_coords_ = Dc_inv_sqrt @ Vt[:k, :].T * sv[:k]
        
        return self.row_coords_
    
    def summary(self):
        """Resumen de la inercia explicada por componente."""
        print("=" * 60)
        print("INERCIA EXPLICADA POR DIMENSIÓN")
        print("=" * 60)
        print(f"{'Dim':>5} {'Eigenvalor':>12} {'Inercia (%)':>13} {'Acumulada (%)':>15}")
        print("-" * 50)
        for i in range(self.n_components):
            ev = self.eigenvalues_[i]
            ei = self.explained_inertia_[i] * 100
            ca = self.cumulative_inertia_[i] * 100
            star = "🔥" if ev > 0.3 else "⚡" if ev > 0.25 else "📊"
            print(f"  {i+1:>3}   {ev:>10.4f}   {ei:>11.2f}%   {ca:>13.2f}%  {star}")

print("✅ Clase MCA_SVD cargada")
print("✅ Librerías importadas correctamente")

In [ ]:
# ============================================================
# CARGA DEL DATASET
# ============================================================
# Cambiar la ruta según donde esté el archivo
df = pd.read_excel("habitos_estudiantes.xlsx")

print("=" * 55)
print("DATASET: HÁBITOS DE SUEÑO Y RENDIMIENTO ESTUDIANTIL")
print("=" * 55)
print(f"🎓 Estudiantes:  {df.shape[0]}")
print(f"📋 Variables:    {df.shape[1]}")
print(f"📊 Columnas:     {list(df.columns)}")
print()
print("Primeras 5 filas:")
display(df.head(5))

In [ ]:
# ============================================================
# EXPLORACIÓN INICIAL
# ============================================================
VARIABLES_ACTIVAS = ['SLEEP', 'CONCENTRACION', 'DESEMPENO',
                     'PUNTUALIDAD', 'FATIGA', 'CALIDAD']
VARIABLES_SUPLEMENTARIAS = ['GENERO', 'CARRERA', 'SEMESTRE', 'BECA']

print("=" * 60)
print("VARIABLES ACTIVAS: distribución de categorías")
print("=" * 60)
for var in VARIABLES_ACTIVAS:
    print(f"\n{'─'*50}")
    print(f"📌 {var}:")
    vc = df[var].value_counts()
    for cat, cnt in vc.items():
        barra = '█' * cnt
        print(f"   {cat:<55} {cnt:>3}  {barra}")

print(f"\n{'─'*60}")
print("VARIABLES SUPLEMENTARIAS:")
for var in VARIABLES_SUPLEMENTARIAS:
    print(f"  {var}: {list(df[var].unique())}")

In [ ]:
# ============================================================
# ANÁLISIS EXPLORATORIO VISUAL
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

paleta = ['#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2', '#b279a2']

for ax, var, color in zip(axes, VARIABLES_ACTIVAS, paleta):
    counts = df[var].value_counts()
    bars = ax.barh(range(len(counts)), counts.values,
                   color=color, alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.set_yticks(range(len(counts)))
    ax.set_yticklabels([str(c)[:40] for c in counts.index], fontsize=9)
    ax.set_title(f"📌 {var}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Frecuencia", fontsize=10)
    for i, (bar, val) in enumerate(zip(bars, counts.values)):
        pct = val / len(df) * 100
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val} ({pct:.0f}%)', va='center', fontsize=9)

plt.suptitle("Distribución de Variables Activas (n=64 estudiantes)",
             fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# 🧹 Sección 3 — Preprocesamiento y Limpieza

## Variables Activas vs Suplementarias

| Tipo | Variables | Rol |
|------|-----------|-----|
| **Activas** | SLEEP, CONCENTRACIÓN, DESEMPEÑO, PUNTUALIDAD, FATIGA, CALIDAD | Construyen los ejes del MCA |
| **Suplementarias** | GÉNERO, CARRERA, SEMESTRE, BECA | Se proyectan para interpretar |

## Estandarización de etiquetas

Las respuestas originales son largas. Las simplificamos para que el biplot sea legible:

| Variable | Original | Simplificado |
|----------|----------|-------------|
| CONCENTRACIÓN | *"Si, en la mayoría de ocasiones"* | *"Mayoría"* |
| DESEMPEÑO | *"Si, mucho"* | *"Mucho"* |
| FATIGA | *"No, nunca lo he pensado"* | *"Nunca"* |
| CALIDAD | *"Si, de forma constante"* | *"Constante"* |

In [ ]:
# ============================================================
# PREPROCESAMIENTO
# ============================================================
df_mca = df.copy()
data_activas = df_mca[VARIABLES_ACTIVAS].copy()

print("=" * 60)
print("PASO 1: VERIFICACIÓN DE DATOS FALTANTES")
print("=" * 60)
faltantes_total = 0
for var in VARIABLES_ACTIVAS:
    n_missing = data_activas[var].isnull().sum()
    estado = "✅ OK" if n_missing == 0 else f"❌ {n_missing} faltantes"
    print(f"  {var:<20}: {estado}")
    faltantes_total += n_missing
if faltantes_total == 0:
    print("\n🎉 Sin datos faltantes en las variables activas.")

print()
print("=" * 60)
print("PASO 2: ESTANDARIZACIÓN DE ETIQUETAS")
print("=" * 60)

# Diccionarios de recodificación
recodes = {
    'CONCENTRACION': {
        'Si, siempre':                                      'Siempre',
        'Si, en la mayoría de ocasiones':                   'Mayoría',
        'Solo en días de exámenes o entregas importantes':  'Solo exámenes',
        'No, no he notado la diferencia':                   'No nota diff'
    },
    'DESEMPENO': {
        'Si, mucho':                       'Mucho',
        'Si pero de manera leve':          'Leve',
        'No noto ninguna diferencia':      'No',
        'Nunca duermo bien':               'No'
    },
    'FATIGA': {
        'No, nunca lo he pensado':         'Nunca',
        'Si, pero nunca lo considere':     'Poco',
        'Si, lo he pensado seriamente':    'Seriamente'
    },
    'CALIDAD': {
        'Si, de forma constante':                          'Constante',
        'Solo en semana de parciales y entregas':          'Solo parciales',
        'No afecta mucho, lo manejo bien':                 'Lo manejo',
        'No afecta en lo absoluto':                        'No afecta'
    }
}

for var, mapping in recodes.items():
    antes = data_activas[var].nunique()
    data_activas[var] = data_activas[var].replace(mapping)
    despues = data_activas[var].nunique()
    print(f"  ✓ {var:<20}: {antes} → {despues} categorías")

print(f"  ✓ SLEEP             : sin cambios (etiquetas ya son cortas)")
print(f"  ✓ PUNTUALIDAD       : sin cambios (etiquetas ya son cortas)")

print()
print("=" * 60)
print("RESULTADO FINAL")
print("=" * 60)
print(f"  Observaciones: {len(data_activas)}")
print(f"  Variables:     {len(VARIABLES_ACTIVAS)}")
print()
for var in VARIABLES_ACTIVAS:
    cats = sorted(data_activas[var].unique())
    print(f"  {var:<20}: {cats}")

data_clean = data_activas.copy()
print("\n✅ datos limpios guardados en data_clean")

In [ ]:
# ============================================================
# TABLAS DE CONTINGENCIA — Relaciones antes del MCA
# ============================================================
# Mostramos cómo DESEMPEÑO se relaciona con otras variables
vars_analizar = ['SLEEP', 'CONCENTRACION', 'FATIGA', 'CALIDAD']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, var in zip(axes, vars_analizar):
    tabla = pd.crosstab(data_clean[var], data_clean['DESEMPENO'],
                        normalize='index') * 100
    sns.heatmap(tabla, annot=True, fmt='.1f', cmap='YlOrRd',
                ax=ax, linewidths=0.5, linecolor='white',
                annot_kws={'size': 9})
    ax.set_title(f'{var} vs DESEMPEÑO\n(% por fila)', fontsize=11, fontweight='bold')
    ax.set_xlabel('DESEMPEÑO', fontsize=10)
    ax.set_ylabel(var, fontsize=10)
    ax.tick_params(axis='both', labelsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

plt.suptitle("Tablas de Contingencia: ¿Cómo se relaciona cada variable con DESEMPEÑO?",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Las tablas muestran el % de estudiantes en cada categoría de DESEMPEÑO,")
print("   dado que pertenecen a esa categoría de la variable de la fila.")
print("   El MCA sintetizará TODAS estas relaciones simultáneamente.")

In [ ]:
# ============================================================
# VISUALIZAR LA MATRIZ INDICADORA (Z)
# ============================================================
Z_demo = pd.get_dummies(data_clean)

print("=" * 60)
print("MATRIZ INDICADORA Z (primeras 8 filas, todas las columnas)")
print("=" * 60)
print(f"Forma de Z: {Z_demo.shape}  →  {len(data_clean)} estudiantes × {Z_demo.shape[1]} categorías")
print(f"Variables: {len(VARIABLES_ACTIVAS)} | Total categorías: {Z_demo.shape[1]}")
print()
print(Z_demo.head(8).astype(int).to_string())
print()
print("📌 Cada fila tiene exactamente UN '1' por variable (columoreado = 6 unos por fila)")
print(f"   Suma por fila (debe ser 6): {Z_demo.sum(axis=1).head(5).values}")

# Visualizar como heatmap
fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(Z_demo.head(20).astype(int), cmap='Blues',
            linewidths=0.3, linecolor='white',
            ax=ax, cbar=False,
            xticklabels=[c.replace('_', '\n') for c in Z_demo.columns],
            yticklabels=[f"Est {i+1}" for i in range(20)])
ax.set_title("Matriz Indicadora Z — Primeros 20 estudiantes\n"
             "(azul = 1: el estudiante tiene esa categoría)",
             fontsize=13, fontweight='bold')
ax.tick_params(axis='x', labelsize=7, rotation=45)

# Separadores por variable
col_start = 0
sep_colors = ['#e74c3c', '#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#1abc9c']
for q_idx, var in enumerate(VARIABLES_ACTIVAS):
    n_cats = data_clean[var].nunique()
    ax.axvline(x=col_start, color=sep_colors[q_idx], linewidth=2.5, alpha=0.8)
    ax.text(col_start + n_cats/2, -1.5, var, ha='center', fontsize=9,
            color=sep_colors[q_idx], fontweight='bold')
    col_start += n_cats

plt.tight_layout()
plt.show()

# ⚙️ Sección 4 — Implementación del MCA

Ahora aplicamos el MCA usando nuestra implementación basada en SVD.

**¿Cuántos componentes usar?**
- En MCA, los porcentajes de inercia explicada son naturalmente más bajos que en PCA
- Buscamos el **codo** en el scree plot (similar a PCA)
- Típicamente se usan **2-4 dimensiones** para visualización

In [ ]:
# ============================================================
# AJUSTAR EL MODELO MCA
# ============================================================
mca = MCA_SVD(n_components=5)
row_coords = mca.fit_transform(data_clean)

print("=" * 60)
print("MCA AJUSTADO EXITOSAMENTE")
print("=" * 60)
print(f"  Individuos:        {mca.n_}")
print(f"  Variables activas: {mca.Q_}")
print(f"  Total categorías:  {mca.K_}")
print(f"  Inercia total:     {mca.K_/mca.Q_ - 1:.4f}")
print()
mca.summary()

# Acceso cómodo a coordenadas
row_df = pd.DataFrame(row_coords,
                      columns=[f"Dim{i+1}" for i in range(5)])
col_df = pd.DataFrame(mca.col_coords_,
                      columns=[f"Dim{i+1}" for i in range(5)],
                      index=mca.col_names_)

print(f"\n✅ row_df: coordenadas de {len(row_df)} estudiantes")
print(f"✅ col_df: coordenadas de {len(col_df)} categorías")

In [ ]:
# ============================================================
# SCREE PLOT Y ANÁLISIS DE INERCIA
# ============================================================
n_dims = 5
eigenvals = mca.eigenvalues_[:n_dims]
inertia_pct = mca.explained_inertia_[:n_dims] * 100
cumul_pct   = mca.cumulative_inertia_[:n_dims] * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Scree Plot ---
ax1 = axes[0]
dims = [f'Dim{i+1}' for i in range(n_dims)]
ax1.plot(dims, eigenvals, 'o-', color='steelblue',
         linewidth=2.5, markersize=11,
         markerfacecolor='white', markeredgewidth=2.5)
ax1.fill_between(dims, eigenvals, alpha=0.12, color='steelblue')
for x, ev in enumerate(eigenvals):
    ax1.annotate(f'λ={ev:.3f}', (x, ev),
                 textcoords='offset points', xytext=(8, 8), fontsize=10)
ax1.set_title("Scree Plot (Eigenvalores)", fontsize=13, fontweight='bold')
ax1.set_ylabel("Eigenvalor", fontsize=12)

# --- Inercia por dimensión ---
ax2 = axes[1]
colors_bar = ['#4c78a8' if i < 2 else '#aec7e8' for i in range(n_dims)]
bars = ax2.bar(dims, inertia_pct, color=colors_bar,
               alpha=0.9, edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, inertia_pct):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.2,
             f'{v:.1f}%', ha='center', fontsize=11, fontweight='bold')
ax2.set_title("Inercia por Dimensión", fontsize=13, fontweight='bold')
ax2.set_ylabel("Inercia explicada (%)", fontsize=12)

# --- Inercia acumulada ---
ax3 = axes[2]
ax3.fill_between(dims, cumul_pct, alpha=0.2, color='darkred')
ax3.plot(dims, cumul_pct, 's-', color='darkred',
         linewidth=2.5, markersize=10,
         markerfacecolor='white', markeredgewidth=2.5)
for x, v in enumerate(cumul_pct):
    ax3.annotate(f'{v:.1f}%', (x, v),
                 textcoords='offset points', xytext=(6, 5),
                 fontsize=10, fontweight='bold', color='darkred')
ax3.axhline(50, color='green', ls='--', alpha=0.7, lw=1.5, label='50%')
ax3.axhline(70, color='orange', ls='--', alpha=0.7, lw=1.5, label='70%')
ax3.set_ylim(0, 115)
ax3.set_title("Inercia Acumulada", fontsize=13, fontweight='bold')
ax3.set_ylabel("Inercia acumulada (%)", fontsize=12)
ax3.legend(fontsize=10)

plt.suptitle("Análisis de Inercia — ¿Cuántas dimensiones conservar?",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 INTERPRETACIÓN:")
print(f"  • Dim1 + Dim2 explican el {cumul_pct[1]:.1f}% de la inercia total.")
print(f"  • Con 4 dimensiones llegamos al {cumul_pct[3]:.1f}%.")
print("  • En MCA, porcentajes del 40-60% con 2 dims ya son aceptables.")
print("  • Usaremos Dim1 y Dim2 para las visualizaciones principales.")

# 🗺️ Sección 5 — Visualizaciones e Interpretación

## El Biplot de MCA

En el mapa factorial del MCA:

| Elemento | Significado |
|----------|-------------|
| **Puntos de categorías** (col_coords) | Posición de cada modalidad en el espacio factorial |
| **Puntos de individuos** (row_coords) | Posición de cada estudiante |
| **Cercanía entre categorías** | Tienden a aparecer en los mismos estudiantes |
| **Cercanía al origen** | Categorías comunes/promedio |
| **Lejanía del origen** | Categorías raras o muy distintivas |

In [ ]:
# ============================================================
# BIPLOT PRINCIPAL: CATEGORÍAS EN EL ESPACIO FACTORIAL
# ============================================================
fig, ax = plt.subplots(figsize=(13, 10))

# Colores por variable
colores_var = {
    'SLEEP':         '#e41a1c',
    'CONCENTRACION': '#377eb8',
    'DESEMPENO':     '#4daf4a',
    'PUNTUALIDAD':   '#ff7f00',
    'FATIGA':        '#984ea3',
    'CALIDAD':       '#a65628'
}

# Dibujar individuos con baja opacidad (fondo)
ax.scatter(row_df['Dim1'], row_df['Dim2'],
           c='lightgray', s=30, alpha=0.5, zorder=1, label='Estudiantes')

# Dibujar categorías
for cat_name, row in col_df.iterrows():
    # Determinar a qué variable pertenece
    var = cat_name.split('_')[0]
    color = colores_var.get(var, 'black')
    label_text = cat_name.replace(var + '_', '')  # Quitar prefijo
    
    ax.scatter(row['Dim1'], row['Dim2'],
               c=color, s=180, alpha=0.9,
               edgecolors='white', linewidth=1.5, zorder=3)
    
    # Etiquetas con ajuste de posición
    offset_x, offset_y = 0.04, 0.04
    ax.annotate(label_text,
                (row['Dim1'], row['Dim2']),
                xytext=(row['Dim1'] + offset_x,
                        row['Dim2'] + offset_y),
                fontsize=9.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',
                          facecolor='white', alpha=0.75, edgecolor=color),
                zorder=4)

ax.axhline(0, color='gray', ls='--', alpha=0.5, lw=1)
ax.axvline(0, color='gray', ls='--', alpha=0.5, lw=1)

ax.set_xlabel(f"Dimensión 1  ({inertia_pct[0]:.1f}% de inercia)",
              fontsize=13, fontweight='bold')
ax.set_ylabel(f"Dimensión 2  ({inertia_pct[1]:.1f}% de inercia)",
              fontsize=13, fontweight='bold')
ax.set_title("Mapa Factorial MCA — Categorías de Variables Activas\n"
             "(puntos del mismo color = misma variable)",
             fontsize=14, fontweight='bold')

# Leyenda de variables
handles = [mpatches.Patch(color=c, label=v)
           for v, c in colores_var.items()]
handles.append(mpatches.Patch(color='lightgray', label='Estudiantes'))
ax.legend(handles=handles, title='Variable', fontsize=10,
          title_fontsize=11, loc='upper left')

# Cuadrantes
for x, y, txt in [(0.97, 0.97, 'Cuadrante I'),
                   (0.03, 0.97, 'Cuadrante II'),
                   (0.03, 0.03, 'Cuadrante III'),
                   (0.97, 0.03, 'Cuadrante IV')]:
    ax.text(x, y, txt, transform=ax.transAxes,
            fontsize=8, color='gray', alpha=0.5,
            ha='right' if x > 0.5 else 'left',
            va='top' if y > 0.5 else 'bottom')

plt.tight_layout()
plt.show()

print("\n📌 CÓMO LEER ESTE MAPA:")
print("  • Categorías del MISMO color: pertenecen a la misma variable")
print("  • Categorías CERCANAS (entre colores): aparecen juntas en los mismos estudiantes")
print("  • Categorías LEJANAS al origen: más raras o más distintivas")
print("  • Categorías cerca del origen: son comunes/promedio")

In [ ]:
# ============================================================
# PROYECCIÓN DE INDIVIDUOS EN ESPACIO FACTORIAL
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Panel 1: Todos los estudiantes ---
ax1 = axes[0]
ax1.scatter(row_df['Dim1'], row_df['Dim2'],
            c='steelblue', s=70, alpha=0.7,
            edgecolors='white', linewidth=0.7)
ax1.axhline(0, color='gray', ls='--', alpha=0.5)
ax1.axvline(0, color='gray', ls='--', alpha=0.5)
ax1.set_title("Nube de Individuos\n(Dim1 vs Dim2)",
              fontsize=12, fontweight='bold')
ax1.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax1.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")

# Centroide general
ax1.scatter(0, 0, marker='+', s=300, c='red',
            linewidth=2.5, label='Origen (promedio)', zorder=5)
ax1.legend(fontsize=9)

# --- Panel 2: Dim1 vs Dim3 ---
ax2 = axes[1]
ax2.scatter(row_df['Dim1'], row_df['Dim3'],
            c='seagreen', s=70, alpha=0.7,
            edgecolors='white', linewidth=0.7)
ax2.axhline(0, color='gray', ls='--', alpha=0.5)
ax2.axvline(0, color='gray', ls='--', alpha=0.5)
ax2.set_title("Nube de Individuos\n(Dim1 vs Dim3)",
              fontsize=12, fontweight='bold')
ax2.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax2.set_ylabel(f"Dim3 ({inertia_pct[2]:.1f}%)")

# --- Panel 3: Dispersión por dimensión ---
ax3 = axes[2]
bp_data = [row_df[f'Dim{i+1}'].values for i in range(5)]
bp = ax3.boxplot(bp_data, patch_artist=True,
                 labels=[f'Dim{i+1}' for i in range(5)])
colors_box = ['#4c78a8', '#f58518', '#54a24b', '#e45756', '#72b7b2']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax3.axhline(0, color='gray', ls='--', alpha=0.5)
ax3.set_title("Distribución de Coordenadas\npor Dimensión",
              fontsize=12, fontweight='bold')
ax3.set_ylabel("Coordenada factorial")

plt.suptitle("Distribución de los 64 Estudiantes en el Espacio MCA",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Estadísticas
print("📊 ESTADÍSTICAS DE DISPERSIÓN POR DIMENSIÓN:")
for i in range(3):
    d = row_df[f'Dim{i+1}']
    print(f"  Dim{i+1}: Media={d.mean():.4f} | Desv.Est={d.std():.3f} "
          f"| Rango=[{d.min():.3f}, {d.max():.3f}]")

In [ ]:
# ============================================================
# VARIABLES SUPLEMENTARIAS: GÉNERO, SEMESTRE, BECA, CARRERA
# ============================================================
data_sup = df.loc[data_clean.index, VARIABLES_SUPLEMENTARIAS].copy()
data_sup['SEMESTRE'] = data_sup['SEMESTRE'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
axes = axes.ravel()

configs = [
    ('GENERO',   'Set1',   'Género'),
    ('BECA',     'Set2',   'Tiene Beca'),
    ('SEMESTRE', 'tab10',  'Semestre académico'),
    ('CARRERA',  'tab20',  'Carrera (top categorías)'),
]

centroides_info = {}

for ax, (var_sup, cmap_name, titulo) in zip(axes, configs):
    cats = sorted(data_sup[var_sup].unique())
    n_cats = len(cats)
    cmap = plt.cm.get_cmap(cmap_name)
    colores_map = {c: cmap(i / max(n_cats-1, 1)) for i, c in enumerate(cats)}
    
    centroides = []
    for cat in cats:
        mask = data_sup[var_sup] == cat
        n_obs = mask.sum()
        if n_obs < 1:
            continue
        color = colores_map[cat]
        
        ax.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
                   c=[color], s=55, alpha=0.65,
                   edgecolors='white', linewidth=0.5,
                   label=f'{cat} (n={n_obs})')
        
        cx = row_df['Dim1'][mask].mean()
        cy = row_df['Dim2'][mask].mean()
        centroides.append((cat, cx, cy, n_obs, color))
        
        # Marcar centroide
        ax.scatter(cx, cy, marker='D', s=180,
                   c=[color], edgecolors='black',
                   linewidth=1.5, zorder=5)
        ax.annotate(f'{cat}', (cx, cy),
                    textcoords='offset points', xytext=(6, 6),
                    fontsize=8, fontweight='bold', color=color,
                    bbox=dict(boxstyle='round,pad=0.15',
                              facecolor='white', alpha=0.7))
    
    centroides_info[var_sup] = centroides
    ax.axhline(0, color='gray', ls='--', alpha=0.5)
    ax.axvline(0, color='gray', ls='--', alpha=0.5)
    ax.set_title(f"Proyección por {titulo}",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)", fontsize=10)
    ax.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)", fontsize=10)
    if n_cats <= 12:
        ax.legend(fontsize=8, ncol=2, loc='lower right')

plt.suptitle("Variables Suplementarias proyectadas en el Espacio MCA\n"
             "(♦ = centroide del grupo)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Imprimir centroides
print("\n📊 CENTROIDES POR VARIABLE SUPLEMENTARIA:")
for var_sup, centroides in centroides_info.items():
    print(f"\n  {var_sup}:")
    for cat, cx, cy, n_obs, _ in centroides:
        dist = np.sqrt(cx**2 + cy**2)
        print(f"    {cat:<40} n={n_obs:>3}  "
              f"Dim1={cx:+.3f}  Dim2={cy:+.3f}  "
              f"dist={dist:.3f}")

In [ ]:
# ============================================================
# BIPLOT COMPLETO: CATEGORÍAS + INDIVIDUOS COLOREADOS POR DESEMPEÑO
# ============================================================
fig, ax = plt.subplots(figsize=(13, 10))

# Colorear individuos por su categoría de DESEMPEÑO
desemp_cats = data_clean['DESEMPENO'].values
desemp_unique = sorted(data_clean['DESEMPENO'].unique())
desemp_colors = {'No': '#e41a1c', 'Leve': '#ff7f00', 'Mucho': '#4daf4a'}

for cat in desemp_unique:
    mask = desemp_cats == cat
    ax.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
               c=desemp_colors[cat], s=80, alpha=0.7,
               edgecolors='white', linewidth=0.7,
               label=f'DESEMPEÑO = {cat}', zorder=2)

# Categorías de variables activas
for cat_name, row in col_df.iterrows():
    var = cat_name.split('_')[0]
    color = colores_var.get(var, 'black')
    label_text = cat_name.replace(var + '_', '')
    
    ax.scatter(row['Dim1'], row['Dim2'],
               c=color, s=280, marker='s',
               edgecolors='black', linewidth=1.2,
               alpha=0.9, zorder=4)
    ax.annotate(f'[{var[:3]}]\n{label_text}',
                (row['Dim1'], row['Dim2']),
                xytext=(row['Dim1'] + 0.03, row['Dim2'] + 0.04),
                fontsize=8.5, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2',
                          facecolor='white', alpha=0.8, edgecolor=color),
                zorder=5)

ax.axhline(0, color='gray', ls='--', alpha=0.4)
ax.axvline(0, color='gray', ls='--', alpha=0.4)

ax.set_xlabel(f"Dimensión 1  ({inertia_pct[0]:.1f}% inercia)",
              fontsize=13, fontweight='bold')
ax.set_ylabel(f"Dimensión 2  ({inertia_pct[1]:.1f}% inercia)",
              fontsize=13, fontweight='bold')
ax.set_title("Biplot MCA — Individuos (por desempeño) + Categorías (■)",
             fontsize=14, fontweight='bold')

# Leyenda individuos
handles_ind = [mpatches.Patch(color=c, label=f'Desempeño={cat}')
               for cat, c in desemp_colors.items()]
handles_var = [mpatches.Patch(color=c, label=f'VAR: {v}')
               for v, c in colores_var.items()]
ax.legend(handles=handles_ind + handles_var,
          fontsize=9, loc='lower left', ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# INTERPRETACIÓN DE LAS DIMENSIONES
# ============================================================
print("=" * 70)
print("INTERPRETACIÓN DE LAS DIMENSIONES FACTORIALES")
print("=" * 70)

for dim_idx in range(2):
    dim_name = f'Dim{dim_idx+1}'
    print(f"\n{'='*65}")
    print(f"DIMENSIÓN {dim_idx+1}  ({inertia_pct[dim_idx]:.1f}% de inercia)")
    print(f"{'='*65}")
    
    # Categorías más positivas
    top_pos = col_df[dim_name].nlargest(5)
    print(f"\n  🔵 Polo POSITIVO (+)")
    for cat, val in top_pos.items():
        var = cat.split('_')[0]
        label = cat.replace(var + '_', '')
        print(f"     [{var}] {label:<30} coordenada = {val:+.3f}")
    
    # Categorías más negativas
    top_neg = col_df[dim_name].nsmallest(5)
    print(f"\n  🔴 Polo NEGATIVO (-)")
    for cat, val in top_neg.items():
        var = cat.split('_')[0]
        label = cat.replace(var + '_', '')
        print(f"     [{var}] {label:<30} coordenada = {val:+.3f}")

# Visualización de barras para interpretación
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, dim_idx in zip(axes, [0, 1]):
    dim_name = f'Dim{dim_idx+1}'
    coords_sorted = col_df[dim_name].sort_values()
    
    colors_bar = ['#e41a1c' if v < 0 else '#4daf4a'
                  for v in coords_sorted.values]
    labels_clean = [f"{idx.split('_')[0][:3]}: {idx.replace(idx.split('_')[0]+'_','')[:25]}"
                    for idx in coords_sorted.index]
    
    bars = ax.barh(range(len(coords_sorted)), coords_sorted.values,
                   color=colors_bar, alpha=0.8,
                   edgecolor='white', linewidth=1)
    ax.set_yticks(range(len(coords_sorted)))
    ax.set_yticklabels(labels_clean, fontsize=9)
    ax.axvline(0, color='black', lw=1.5)
    ax.set_title(f"Coordenadas en Dimensión {dim_idx+1}"
                 f" ({inertia_pct[dim_idx]:.1f}%)",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel("Coordenada factorial", fontsize=11)
    
    # Anotaciones
    for bar, val in zip(bars, coords_sorted.values):
        ax.text(val + (0.01 if val >= 0 else -0.01),
                bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center',
                ha='left' if val >= 0 else 'right',
                fontsize=8)

plt.suptitle("Contribución de cada Categoría a las 2 Primeras Dimensiones",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 🧑‍🎓 Sección 6 — Perfiles Estudiantiles y Conclusiones

## 6.1 Perfiles identificados por el MCA

A partir del mapa factorial, identificamos **3 perfiles principales** de estudiantes:

| Perfil | Ubicación en mapa | Características |
|--------|------------------|-----------------|
| **Perfil 1: Alto Rendimiento** | Extremo + Dim1 | Sueño frecuente, alta concentración, buen desempeño, poca fatiga |
| **Perfil 2: Estudiante en Riesgo** | Extremo − Dim1 | Sueño irregular, baja concentración, mayor fatiga, irregular puntualidad |
| **Perfil 3: Adaptativo (intermedio)** | Centro del mapa | Maneja bien a pesar de sueño irregular; compensa con esfuerzo |

In [ ]:
# ============================================================
# IDENTIFICACIÓN DE PERFILES ESTUDIANTILES
# ============================================================

# Definir perfiles por umbral en Dim1
dim1_q33 = row_df['Dim1'].quantile(0.33)
dim1_q66 = row_df['Dim1'].quantile(0.66)

def asignar_perfil(dim1):
    if dim1 >= dim1_q66:
        return 'Perfil 1: Alto rendimiento'
    elif dim1 <= dim1_q33:
        return 'Perfil 2: En riesgo'
    else:
        return 'Perfil 3: Adaptativo'

perfiles = row_df['Dim1'].apply(asignar_perfil)

print("=" * 60)
print("DISTRIBUCIÓN DE PERFILES ESTUDIANTILES")
print("=" * 60)
print(perfiles.value_counts())

# Características promedio por perfil
print("\n=" * 60)
print("CATEGORÍA MÁS FRECUENTE POR VARIABLE Y PERFIL")
print("=" * 60)
data_con_perfil = data_clean.copy()
data_con_perfil['PERFIL'] = perfiles.values

for perfil in ['Perfil 1: Alto rendimiento',
               'Perfil 2: En riesgo',
               'Perfil 3: Adaptativo']:
    subset = data_con_perfil[data_con_perfil['PERFIL'] == perfil]
    print(f"\n  {perfil} (n={len(subset)}):")
    for var in VARIABLES_ACTIVAS:
        top = subset[var].value_counts().index[0]
        cnt = subset[var].value_counts().iloc[0]
        pct = cnt / len(subset) * 100
        print(f"    {var:<20}: {top:<35} ({pct:.0f}%)")

# VISUALIZACIÓN DE PERFILES
colores_perfil = {
    'Perfil 1: Alto rendimiento': '#4daf4a',
    'Perfil 2: En riesgo':        '#e41a1c',
    'Perfil 3: Adaptativo':       '#377eb8'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Mapa factorial con perfiles
ax1 = axes[0]
for perfil, color in colores_perfil.items():
    mask = perfiles == perfil
    ax1.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
                c=color, s=90, alpha=0.8,
                edgecolors='white', linewidth=0.7,
                label=perfil)
    # Centroide del perfil
    cx = row_df['Dim1'][mask].mean()
    cy = row_df['Dim2'][mask].mean()
    ax1.scatter(cx, cy, marker='*', s=450,
                c=color, edgecolors='black', linewidth=1.5,
                zorder=5)

# Superponer categorías clave
cats_clave = ['SLEEP_Frecuentemente', 'SLEEP_Siempre',
              'DESEMPENO_Mucho', 'DESEMPENO_No',
              'FATIGA_Nunca', 'FATIGA_Seriamente',
              'CONCENTRACION_Siempre', 'CONCENTRACION_Solo exámenes']
for cat in cats_clave:
    if cat in col_df.index:
        var = cat.split('_')[0]
        label = cat.replace(var + '_', '')
        ax1.scatter(col_df.loc[cat, 'Dim1'],
                    col_df.loc[cat, 'Dim2'],
                    marker='D', s=150,
                    c=colores_var[var],
                    edgecolors='black', linewidth=1.2,
                    alpha=0.9, zorder=6)
        ax1.annotate(f'{label}', (col_df.loc[cat,'Dim1'],
                                   col_df.loc[cat,'Dim2']),
                     fontsize=8, color=colores_var[var],
                     fontweight='bold',
                     textcoords='offset points',
                     xytext=(5, 5),
                     bbox=dict(boxstyle='round,pad=0.15',
                               facecolor='white', alpha=0.75))

ax1.axhline(0, color='gray', ls='--', alpha=0.4)
ax1.axvline(0, color='gray', ls='--', alpha=0.4)
ax1.axvline(dim1_q33, color='gray', ls=':', alpha=0.6, lw=1.5)
ax1.axvline(dim1_q66, color='gray', ls=':', alpha=0.6, lw=1.5)
ax1.set_title("Perfiles Estudiantiles en el Espacio MCA",
              fontsize=12, fontweight='bold')
ax1.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax1.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")
ax1.legend(fontsize=9)

# Barra comparativa de perfiles
ax2 = axes[1]
perfiles_counts = perfiles.value_counts()
bars = ax2.bar(range(len(perfiles_counts)),
               perfiles_counts.values,
               color=[colores_perfil[k] for k in perfiles_counts.index],
               alpha=0.85, edgecolor='white', linewidth=2)
ax2.set_xticks(range(len(perfiles_counts)))
ax2.set_xticklabels([p.replace('Perfil ', 'P').split(':')[0] + ':\n'
                     + p.split(':')[1].strip()
                     for p in perfiles_counts.index],
                    fontsize=10)
for bar, val in zip(bars, perfiles_counts.values):
    pct = val / len(perfiles) * 100
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f'{val}\n({pct:.0f}%)',
             ha='center', fontsize=12, fontweight='bold')
ax2.set_title("Distribución de Perfiles", fontsize=12, fontweight='bold')
ax2.set_ylabel("Número de estudiantes", fontsize=11)

plt.suptitle("Identificación de Perfiles Estudiantiles mediante MCA",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# RESUMEN VISUAL FINAL: TODO EN UNA FIGURA
# ============================================================
fig = plt.figure(figsize=(18, 13))
gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 1. Scree plot
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(1, 6), eigenvals, 'o-',
         color='steelblue', lw=2.5, ms=9,
         mfc='white', mew=2.5)
ax1.set_title("(1) Scree Plot", fontweight='bold', fontsize=11)
ax1.set_xticks(range(1, 6))
ax1.set_xticklabels([f'Dim{i}' for i in range(1, 6)])
ax1.set_ylabel("Eigenvalor")

# 2. Inercia acumulada
ax2 = fig.add_subplot(gs[0, 1])
ax2.fill_between(range(1, 6), cumul_pct, alpha=0.2, color='darkred')
ax2.plot(range(1, 6), cumul_pct, 's-',
         color='darkred', lw=2.5, ms=9,
         mfc='white', mew=2.5)
for i, v in enumerate(cumul_pct):
    ax2.annotate(f'{v:.0f}%', (i+1, v),
                 textcoords='offset points', xytext=(5, 5),
                 fontsize=9, color='darkred', fontweight='bold')
ax2.axhline(50, color='green', ls='--', alpha=0.6, lw=1.5)
ax2.set_title("(2) Inercia Acumulada", fontweight='bold', fontsize=11)
ax2.set_xticks(range(1, 6))
ax2.set_xticklabels([f'Dim{i}' for i in range(1, 6)])
ax2.set_ylim(0, 110)
ax2.set_ylabel("Inercia acumulada (%)")

# 3. Biplot categorías principales
ax3 = fig.add_subplot(gs[0, 2])
for cat_name, row in col_df.iterrows():
    var = cat_name.split('_')[0]
    color = colores_var.get(var, 'black')
    ax3.scatter(row['Dim1'], row['Dim2'],
                c=color, s=80, alpha=0.9,
                edgecolors='white', lw=1)
    label_t = cat_name.replace(var + '_', '')[:12]
    ax3.annotate(label_t, (row['Dim1'], row['Dim2']),
                 xytext=(3, 3), textcoords='offset points',
                 fontsize=7, color=color, fontweight='bold')
ax3.axhline(0, color='gray', ls='--', alpha=0.4)
ax3.axvline(0, color='gray', ls='--', alpha=0.4)
ax3.set_title("(3) Mapa de Categorías", fontweight='bold', fontsize=11)
ax3.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax3.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")

# 4. Perfiles
ax4 = fig.add_subplot(gs[1, 0])
for perfil, color in colores_perfil.items():
    mask = perfiles == perfil
    ax4.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
                c=color, s=50, alpha=0.8,
                edgecolors='white', lw=0.5,
                label=perfil.split(':')[0])
ax4.axhline(0, color='gray', ls='--', alpha=0.4)
ax4.axvline(0, color='gray', ls='--', alpha=0.4)
ax4.set_title("(4) Perfiles de Estudiantes", fontweight='bold', fontsize=11)
ax4.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax4.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")
ax4.legend(fontsize=8)

# 5. Proyección por Género
ax5 = fig.add_subplot(gs[1, 1])
for cat, color in zip(['Femenino', 'Masculino'],
                       ['#e41a1c', '#377eb8']):
    mask = data_sup['GENERO'] == cat
    ax5.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
                c=color, s=50, alpha=0.7,
                edgecolors='white', lw=0.5, label=cat)
ax5.axhline(0, color='gray', ls='--', alpha=0.4)
ax5.axvline(0, color='gray', ls='--', alpha=0.4)
ax5.set_title("(5) Proyección por Género", fontweight='bold', fontsize=11)
ax5.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax5.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")
ax5.legend(fontsize=9)

# 6. Proyección por Beca
ax6 = fig.add_subplot(gs[1, 2])
for cat, color in zip(['No', 'Si'],
                       ['#ff7f00', '#4daf4a']):
    mask = data_sup['BECA'] == cat
    ax6.scatter(row_df['Dim1'][mask], row_df['Dim2'][mask],
                c=color, s=50, alpha=0.7,
                edgecolors='white', lw=0.5,
                label=f'Beca={cat}')
ax6.axhline(0, color='gray', ls='--', alpha=0.4)
ax6.axvline(0, color='gray', ls='--', alpha=0.4)
ax6.set_title("(6) Proyección por Beca", fontweight='bold', fontsize=11)
ax6.set_xlabel(f"Dim1 ({inertia_pct[0]:.1f}%)")
ax6.set_ylabel(f"Dim2 ({inertia_pct[1]:.1f}%)")
ax6.legend(fontsize=9)

plt.suptitle("Resumen Visual Completo — MCA: Hábitos de Sueño y Rendimiento",
             fontsize=16, fontweight='bold')
plt.show()

print("\n" + "="*65)
print("   FIN — ANÁLISIS DE CORRESPONDENCIA MÚLTIPLE (MCA)")
print("="*65)
print()
print("  📌 Conceptos clave:")
print("  1. MCA es la extensión de PCA para variables categóricas")
print("  2. Usa la descomposición SVD sobre la matriz indicadora Z")
print("  3. Los mapas factoriales muestran relaciones entre categorías")
print("  4. Categorías cercanas aparecen juntas en los mismos individuos")
print("  5. Variables suplementarias ayudan a interpretar sin sesgrar")
print(f"  6. Se identificaron 3 perfiles estudiantiles distintos")

# 🏁 Conclusiones

## Hallazgos del MCA

- Las **2 primeras dimensiones** explican un porcentaje significativo de la inercia, mostrando las principales oposiciones entre categorías.
- La **Dimensión 1** refleja un eje de **bienestar académico**: sueño frecuente y buen desempeño vs. irregularidad y fatiga.
- La **Dimensión 2** refleja diferencias en el **manejo del estrés y la regularidad**.

## Perfiles Identificados

| Perfil | Características principales |
|--------|-----------------------------|
| **Alto Rendimiento** | Duerme frecuente/siempre, alta concentración, desempeño alto, baja fatiga |
| **En Riesgo** | Sueño irregular, menor concentración, fatiga elevada, puntualidad baja |
| **Adaptativo** | Perfil intermedio, compensa con estrategias propias |

## Limitaciones

- Muestra de **solo 64 estudiantes** (baja potencia estadística)
- Algunas carreras tienen **1 solo estudiante** → centroides no representativos
- Variables **autodeclaradas** → posible sesgo de deseabilidad social
- **Estudio transversal** → no muestra evolución temporal

## ¿Cuándo usar MCA?

| Situación | MCA? |
|-----------|------|
| Variables categóricas múltiples (encuestas) | ✅ Ideal |
| Quiero un mapa visual de categorías | ✅ Ideal |
| Identificar perfiles/clusters de individuos | ✅ Sí |
| Variables numéricas continuas | ❌ Usar PCA |
| Solo 2 variables categóricas | ❌ Usar Chi-cuadrado / CA simple |